In [1]:

import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get("Australian_Rain", version="1")

df = pd.read_csv(data_asset.path)
df

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
Overriding of current TracerProvider is not allowed
Overriding

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145455,2017-06-21,Uluru,2.8,23.4,0.0,NaN,NaN,E,31.0,SE,...,51.0,24.0,1024.6,1020.3,NaN,NaN,10.1,22.4,No,No
145456,2017-06-22,Uluru,3.6,25.3,0.0,NaN,NaN,NNW,22.0,SE,...,56.0,21.0,1023.5,1019.1,NaN,NaN,10.9,24.5,No,No
145457,2017-06-23,Uluru,5.4,26.9,0.0,NaN,NaN,N,37.0,SE,...,53.0,24.0,1021.0,1016.8,NaN,NaN,12.5,26.1,No,No
145458,2017-06-24,Uluru,7.8,27.0,0.0,NaN,NaN,SE,28.0,SSE,...,51.0,24.0,1019.4,1016.5,3.0,2.0,15.1,26.0,No,No


In [2]:
import pandas as pd
import numpy as np
import os
import joblib

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# =========================================================
# 1. Cargar dataset desde Azure ML
# =========================================================
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get("Australian_Rain", version="1")

df = pd.read_csv(data_asset.path)

print("Dimensiones del dataset:", df.shape)
display(df.head())
display(df.info())


Found the config file in: /config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Dimensiones del dataset: (145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  object 
 1   Location       145460 non-null  object 
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object 
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object 
 10  WindDir3pm     141232 non-null  object 
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null

None

In [3]:

# =========================================================
# 2. Limpieza inicial
# =========================================================
df = df.drop_duplicates().copy()

# Estandarizar posibles faltantes
missing_tokens = ["", " ", "NA", "N/A", "na", "null", "NULL", "?", "None"]
df = df.replace(missing_tokens, np.nan)

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

print("\nValores faltantes por columna:")
display(df.isna().sum().sort_values(ascending=False))



Valores faltantes por columna:


Sunshine         69835
Evaporation      62790
Cloud3pm         59358
Cloud9am         55888
Pressure9am      15065
Pressure3pm      15028
WindDir9am       10566
WindGustDir      10326
WindGustSpeed    10263
Humidity3pm       4507
WindDir3pm        4228
Temp3pm           3609
RainTomorrow      3267
Rainfall          3261
RainToday         3261
WindSpeed3pm      3062
Humidity9am       2654
Temp9am           1767
WindSpeed9am      1767
MinTemp           1485
MaxTemp           1261
Location             0
Date                 0
dtype: int64

In [4]:

# =========================================================
# 3. Preparación del target
# =========================================================
target_col = "RainTomorrow"

# Eliminar filas donde falte la variable objetivo
df = df.dropna(subset=[target_col]).copy()

# Convertir target a binario
df[target_col] = df[target_col].astype(str).str.strip().str.lower()
df[target_col] = df[target_col].map({"yes": 1, "no": 0})

# Eliminar filas que no hayan podido mapearse
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

# También convertir RainToday si existe
if "RainToday" in df.columns:
    df["RainToday"] = df["RainToday"].astype(str).str.strip().str.lower()
    df["RainToday"] = df["RainToday"].map({"yes": 1, "no": 0})

print("\nDistribución de clases:")
print(df[target_col].value_counts())
print(df[target_col].value_counts(normalize=True))



Distribución de clases:
0    110316
1     31877
Name: RainTomorrow, dtype: int64
0    0.775819
1    0.224181
Name: RainTomorrow, dtype: float64


In [5]:

# =========================================================
# 4. Ingeniería básica de características
# =========================================================
# Convertir Date en variables útiles
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df = df.drop(columns=["Date"])


In [6]:

# =========================================================
# 5. Separar X e y
# =========================================================
X = df.drop(columns=[target_col])
y = df[target_col]

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("\nColumnas numéricas:", numeric_features)
print("\nColumnas categóricas:", categorical_features)



Columnas numéricas: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'Year', 'Month', 'Day']

Columnas categóricas: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']


In [7]:

# =========================================================
# 6. Dividir datos en train, validation y test
# =========================================================
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,   # 0.25 de 0.80 = 0.20 total
    stratify=y_trainval,
    random_state=42
)

print("\nShapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)



Shapes:
Train: (85315, 24) (85315,)
Val: (28439, 24) (28439,)
Test: (28439, 24) (28439,)


In [8]:

# =========================================================
# 7. Preprocesamiento
# =========================================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [9]:

# =========================================================
# 8. Pipeline con modelo
# =========================================================
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])


In [10]:

# =========================================================
# 9. Ajuste de hiperparámetros con validación cruzada
# =========================================================
param_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="f1",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

search.fit(X_train, y_train)

print("\nMejores hiperparámetros:")
print(search.best_params_)
print("\nMejor score de validación cruzada:", search.best_score_)

best_model = search.best_estimator_


Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END model__max_depth=10, model__max_features=log2, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=100; total time=  16.7s
[CV] END model__max_depth=10, model__max_features=log2, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=100; total time=  16.0s
[CV] END model__max_depth=10, model__max_features=log2, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  15.5s
[CV] END model__max_depth=10, model__max_features=None, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=300; total time=14.6min
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=2, model__n_estimators=100; total time= 2.4min
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=2, model__n_estimators=100; total time= 2.3min
[CV

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from usi

In [11]:

# =========================================================
# 10. Función de evaluación
# =========================================================
def evaluate_model(model, X_data, y_true, dataset_name="dataset"):
    y_pred = model.predict(X_data)
    y_proba = model.predict_proba(X_data)[:, 1]

    metrics = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba)
    }

    print(f"\n===== {dataset_name.upper()} =====")
    for k, v in metrics.items():
        if k != "dataset":
            print(f"{k}: {v:.4f}")

    print("\nMatriz de confusión:")
    print(confusion_matrix(y_true, y_pred))

    print("\nReporte de clasificación:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return metrics


In [12]:

# =========================================================
# 11. Evaluación en train, val y test
# =========================================================
train_metrics = evaluate_model(best_model, X_train, y_train, "train")
val_metrics = evaluate_model(best_model, X_val, y_val, "validation")
test_metrics = evaluate_model(best_model, X_test, y_test, "test")



===== TRAIN =====
accuracy: 0.9370
precision: 0.8220
recall: 0.9179
f1: 0.8673
roc_auc: 0.9845

Matriz de confusión:
[[62388  3801]
 [ 1571 17555]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0     0.9754    0.9426    0.9587     66189
           1     0.8220    0.9179    0.8673     19126

    accuracy                         0.9370     85315
   macro avg     0.8987    0.9302    0.9130     85315
weighted avg     0.9410    0.9370    0.9382     85315


===== VALIDATION =====
accuracy: 0.8392
precision: 0.6364
recall: 0.6600
f1: 0.6480
roc_auc: 0.8809

Matriz de confusión:
[[19659  2404]
 [ 2168  4208]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0     0.9007    0.8910    0.8958     22063
           1     0.6364    0.6600    0.6480      6376

    accuracy                         0.8392     28439
   macro avg     0.7685    0.7755    0.7719     28439
weighted avg     0.8414    0.8392    0.8403 

In [13]:

# =========================================================
# 12. Revisión básica de sobreajuste
# =========================================================
print("\n===== REVISIÓN DE SOBREAJUSTE =====")
print(f"F1 Train: {train_metrics['f1']:.4f}")
print(f"F1 Validation: {val_metrics['f1']:.4f}")
print(f"F1 Test: {test_metrics['f1']:.4f}")

gap_train_val = train_metrics["f1"] - val_metrics["f1"]
gap_train_test = train_metrics["f1"] - test_metrics["f1"]

print(f"Gap Train-Val: {gap_train_val:.4f}")
print(f"Gap Train-Test: {gap_train_test:.4f}")

if gap_train_val > 0.05 or gap_train_test > 0.05:
    print("Posible sobreajuste: el modelo rinde bastante mejor en entrenamiento que en validación/test.")
else:
    print("No hay señales fuertes de sobreajuste según el gap de F1.")

# También revisar CV
cv_results = pd.DataFrame(search.cv_results_)
best_idx = search.best_index_

mean_train_cv = cv_results.loc[best_idx, "mean_train_score"]
mean_valid_cv = cv_results.loc[best_idx, "mean_test_score"]
std_valid_cv = cv_results.loc[best_idx, "std_test_score"]

print(f"\nMean train score CV: {mean_train_cv:.4f}")
print(f"Mean validation score CV: {mean_valid_cv:.4f}")
print(f"Std validation score CV: {std_valid_cv:.4f}")



===== REVISIÓN DE SOBREAJUSTE =====
F1 Train: 0.8673
F1 Validation: 0.6480
F1 Test: 0.6595
Gap Train-Val: 0.2193
Gap Train-Test: 0.2078
Posible sobreajuste: el modelo rinde bastante mejor en entrenamiento que en validación/test.

Mean train score CV: 0.8678
Mean validation score CV: 0.6541
Std validation score CV: 0.0021


In [14]:

# =========================================================
# 13. Resumen final
# =========================================================
summary = pd.DataFrame([train_metrics, val_metrics, test_metrics])
display(summary)


,dataset,accuracy,precision,recall,f1,roc_auc
0,train,0.937033,0.822017,0.917861,0.867299,0.984488
1,validation,0.839235,0.636419,0.659975,0.647983,0.880920
2,test,0.844474,0.647566,0.671843,0.659481,0.886934


In [15]:

# =========================================================
# 14. Guardar el modelo
# =========================================================
os.makedirs("outputs", exist_ok=True)
joblib.dump(best_model, "outputs/weatherAUS_random_forest.pkl")
print("\nModelo guardado en outputs/weatherAUS_random_forest.pkl")


Modelo guardado en outputs/weatherAUS_random_forest.pkl
